In [2]:
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [1]:
limma_results = "/output/limma/"

__GSEA function__

In [ ]:
def gsea_analysis(limma, folder_name,
                  gene_sets="GO_Biological_Process_2023",
                  pval_thresh=0.05):

    # ---------------------------
    # Prepare output folder
    # ---------------------------
    outdir = os.path.join("output", "GSEA", folder_name)
    os.makedirs(outdir, exist_ok=True)

    res_df = limma.copy()

    # ---------------------------
    # Clean data
    # ---------------------------
    res_df = res_df.replace([np.inf, -np.inf], np.nan)
    res_df = res_df.dropna(subset=["log2FC", "adj_pvalue", "Protein"])

    # Avoid log10(0)
    res_df = res_df[res_df["adj_pvalue"] > 0]

    # ---------------------------
    # Ranking metric
    # ---------------------------
    res_df["rank_metric"] = -np.log10(res_df["adj_pvalue"]) * np.sign(res_df["log2FC"])

    # ---------------------------
    # Build ranked list
    # ---------------------------
    ranking = res_df[["Protein", "rank_metric"]].copy()

    # Remove duplicates
    ranking = ranking.drop_duplicates(subset="Protein")

    # Sort descending
    ranking = ranking.sort_values("rank_metric", ascending=False)
    ranking = ranking.set_index("Protein")["rank_metric"]

    # Save ranking file
    rnk_path = os.path.join(outdir, "ranking.rnk")
    ranking.to_csv(rnk_path, sep="\t", index=False, header=False)

    # ---------------------------
    # Load gene sets properly
    # ---------------------------

    if isinstance(gene_sets, str):
        print(f"📚 Loading gene set: {gene_sets}")
        gene_sets = gp.get_library(name=gene_sets, organism="Human")

    # ---------------------------
    # Run GSEA prerank
    # ---------------------------
    gsea_res = gp.prerank(
        rnk=ranking,
        min_size=5,
        gene_sets=gene_sets,
        organism="Human",
        outdir=outdir,
        seed=42,
        verbose=True
    )

    # ---------------------------
    # Extract results
    # ---------------------------
    gsea_df = gsea_res.res2d.copy()

    # ---------------------------
    # Add metadata if available
    # ---------------------------
    if "comparison" in res_df.columns:
        gsea_df["comparison"] = res_df["comparison"].iloc[0]

    if "confounders" in res_df.columns:
        gsea_df["confounders"] = res_df["confounders"].iloc[0]

    # ---------------------------
    # Filter significant pathways
    # ---------------------------
    sig = gsea_df[gsea_df["FDR q-val"] < pval_thresh].copy()

    # Save results
    gsea_df.to_csv(os.path.join(outdir, "gsea_full_results.tsv"), sep="\t", index=False)
    sig.to_csv(os.path.join(outdir, "gsea_significant.tsv"), sep="\t", index=False)

    print(f"\n✅ GSEA completed: {folder_name}")
    print(f"Total pathways: {gsea_df.shape[0]}")
    print(f"Significant pathways (FDR < {pval_thresh}): {sig.shape[0]}")

    return gsea_df

_looping through the limma outputs_

In [ ]:
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-M_vs_F_all.tsv", sep="\t"),
              folder_name="M_vs_F_all",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-M_vs_F_T0_notox.tsv", sep="\t"),
              folder_name="M_vs_F_T0_notox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-M_vs_F_T0_tox.tsv", sep="\t"),
              folder_name="M_vs_F_T0_tox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T1_vs_T0_all.tsv", sep="\t"),
              folder_name="T1_vs_T0_all",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T1_vs_T0_F.tsv", sep="\t"),
              folder_name="T1_vs_T0_F",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T1_vs_T0_M.tsv", sep="\t"),
              folder_name="T1_vs_T0_M",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T1_vs_T0_notox.tsv", sep="\t"),
              folder_name="T1_vs_T0_notox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T1_vs_T0_tox.tsv", sep="\t"),
              folder_name="T1_vs_T0_tox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T2_vs_T0_all.tsv", sep="\t"),
              folder_name="T2_vs_T0_all",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T2_vs_T0_F_tox.tsv", sep="\t"),
              folder_name="T2_vs_T0_F_tox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T2_vs_T0_F.tsv", sep="\t"),
              folder_name="T2_vs_T0_F",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T2_vs_T0_M_tox.tsv", sep="\t"),
              folder_name="T2_vs_T0_M_tox",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-T2_vs_T0_M.tsv", sep="\t"),
              folder_name="T2_vs_T0_M",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)
gsea_analysis(limma=pd.read_csv("output/limma/limma_diff_prot-tox_vs_notox_T0.tsv", sep="\t"),
              folder_name="tox_vs_notox_T0",
              gene_sets="GO_Biological_Process_2023",
              pval_thresh=0.05)

Protein
CDHR2     4.854572
GGT1      1.231901
GGT7      1.231901
MMP3      1.231901
NCAM1     1.231901
            ...   
IMMT     -0.443159
APLP2    -0.758119
PSMD13   -1.231901
LEP      -1.231901
PZP      -2.338384
Name: rank_metric, Length: 1999, dtype: float64
📚 Loading gene set: GO_Biological_Process_2023


In [ ]:
res = pd.read_csv("output/GSEA_Sex/gseapy.gene_set.prerank.report.csv")
res = res.sort_values("FDR q-val", ascending=True)
res

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Positive Regulation Of Protein Localization To...,-0.662155,-1.991728,0.005376,0.090291,0.060,6/19,6.84%,CCT4;CCT5;CCT6A;TGFB1;CCT2;TCP1
1,prerank,Regulation Of Protein Binding (GO:0043393),-0.613416,-1.860509,0.000000,0.151661,0.164,8/20,16.85%,APP;PRKACA;RACK1;CFHR2;ADD2;RAN;BAX;CAMK1
4,prerank,Positive Regulation Of Tumor Necrosis Factor P...,-0.535370,-1.587368,0.028571,0.546685,0.660,7/19,17.25%,APP;MMP8;THBS1;HSPB1;SPN;HMGB1;LCP1
5,prerank,Positive Regulation Of Tumor Necrosis Factor S...,-0.535370,-1.587368,0.028571,0.546685,0.660,7/19,17.25%,APP;MMP8;THBS1;HSPB1;SPN;HMGB1;LCP1
3,prerank,Positive Regulation Of Kinase Activity (GO:003...,0.641238,1.648552,0.002381,0.592296,0.586,16/32,20.52%,HLA-DRB1;ROR1;FGFR3;EPHA1;GAS6;MET;MERTK;PTPRC...
...,...,...,...,...,...,...,...,...,...,...
212,prerank,Cellular Response To Molecule Of Bacterial Ori...,0.402888,0.936833,0.579011,1.000000,1.000,7/17,27.57%,VIM;SERPINE1;SIRPA;LILRB2;AXL;PDCD1LG2;RHOA
213,prerank,Regulation Of Reactive Oxygen Species Metaboli...,0.403344,0.925718,0.593506,1.000000,1.000,6/16,22.38%,SNCA;RAC1;SORD;DCXR;PDGFRB;EIF6
214,prerank,Organelle Assembly (GO:0070925),0.360830,0.925333,0.601645,1.000000,1.000,6/32,14.44%,EHD1;ACTR2;ACTR3;CEP126;CORO1A;EHD3
246,prerank,Neuron Development (GO:0048666),0.382132,0.869684,0.660622,1.000000,1.000,3/17,7.70%,EHD1;L1CAM;CLN5


In [12]:
res = res.sort_values("NES", ascending=False)
res

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
2,prerank,Regulation Of Kinase Activity (GO:0043549),0.681700,1.659708,0.003690,1.000000,0.536,14/23,22.99%,HLA-DRB1;ROR1;TTN;FGFR3;EPHA1;MET;MERTK;EGFR;F...
3,prerank,Positive Regulation Of Kinase Activity (GO:003...,0.641238,1.648552,0.002381,0.592296,0.586,16/32,20.52%,HLA-DRB1;ROR1;FGFR3;EPHA1;GAS6;MET;MERTK;PTPRC...
6,prerank,Cell-Cell Adhesion Via Plasma-Membrane Adhesio...,0.566806,1.575532,0.008899,1.000000,0.891,28/52,29.58%,HMCN2;CDH23;CDH11;L1CAM;SDK1;SELP;PCDHGC3;CDH2...
7,prerank,Homophilic Cell Adhesion Via Plasma Membrane A...,0.623340,1.560083,0.014337,0.934940,0.927,18/27,27.67%,HMCN2;L1CAM;SDK1;CDH2;PTK7;HMCN1;PTPRM;PLXNB2;...
8,prerank,Synapse Organization (GO:0050808),0.599059,1.519417,0.025362,1.000000,0.979,9/28,11.42%,HMCN2;AGRN;CAST;L1CAM;SDK1;NRCAM;CDH2;PTK7;SNCA
...,...,...,...,...,...,...,...,...,...,...
10,prerank,Negative Regulation Of Proteolysis (GO:0045861),-0.488259,-1.476588,0.059091,0.848737,0.872,4/18,6.29%,SERPINE2;CSTB;CST4;SERPINB3
4,prerank,Positive Regulation Of Tumor Necrosis Factor P...,-0.535370,-1.587368,0.028571,0.546685,0.660,7/19,17.25%,APP;MMP8;THBS1;HSPB1;SPN;HMGB1;LCP1
5,prerank,Positive Regulation Of Tumor Necrosis Factor S...,-0.535370,-1.587368,0.028571,0.546685,0.660,7/19,17.25%,APP;MMP8;THBS1;HSPB1;SPN;HMGB1;LCP1
1,prerank,Regulation Of Protein Binding (GO:0043393),-0.613416,-1.860509,0.000000,0.151661,0.164,8/20,16.85%,APP;PRKACA;RACK1;CFHR2;ADD2;RAN;BAX;CAMK1


_Male-enriched GSEA_

In [13]:
male_rank = ranked_series[ranked_series > 0].sort_values(ascending=False)
pre_male = gp.prerank(
    rnk=male_rank,
    gene_sets="GO_Biological_Process_2023",
    organism="Human",
    outdir="output/GSEA_Sex_Male",
    seed=42
)

2026-04-16 11:14:33,208 [WARNING] Duplicated values found in preranked stats: 5.43% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


_Female-enriched GSEA_

In [14]:
female_rank = ranked_series[ranked_series < 0].sort_values(ascending=True)
female_rank = female_rank.abs() * -1  # keep direction consistent
pre_male = gp.prerank(
    rnk=male_rank,
    gene_sets="GO_Biological_Process_2023",
    organism="Human",
    outdir="output/GSEA_Sex_Female",
    seed=42
)

2026-04-16 11:14:33,435 [WARNING] Duplicated values found in preranked stats: 5.43% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


__Visualizations__

In [15]:
male_res = pd.read_csv("output/GSEA_Sex_Male/gseapy.gene_set.prerank.report.csv")
female_res = pd.read_csv("output/GSEA_Sex_Female/gseapy.gene_set.prerank.report.csv")

In [16]:
# Filter significant pathways
male_res = male_res[male_res["FDR q-val"] < 0.6]
male_res
female_res = female_res[female_res["FDR q-val"] < 0.6]
female_res


# Select top N
top_n = 10

male_top = male_res.sort_values("NES", ascending=False).head(top_n)
female_top = female_res.sort_values("NES", ascending=False).head(top_n)

In [17]:
male_top["Group"] = "Male"
female_top["Group"] = "Female"
plot_df = pd.concat([male_top, female_top])
plot_df["log10_FDR"] = -np.log10(plot_df["FDR q-val"])